# Data Exploration

We're building a retrieval system using the **TREC-COVID** benchmark dataset built during the Covid-19 pandemic, a collection of medical papers of various lengths on COVID-19 and related coronaviruses. Each document contains 50 queries observed through three categories: **Title**, **Description**, and **Narrative**.

## Data Exploration Steps

Our data exploration contains a few key steps:

1. **Analyzing Query Verbosity**
   First, we explore the queries at three levels of verbosity: 
   * Short titles (averaging ~3 words)
   * Longer descriptions (averaging ~11 words)
   * More detailed narratives (averaging ~24 words)
   
   This builds up our experimental lens: *does giving more context and information regarding queries yield better results for information retrieval in the medical domain?*

2. **Preprocessing & Spell Check Analysis**
   Next, we preprocess those queries where we turn them to lowercase, remove punctuation, perform stemming, and remove stop words. We performed a spell-checking analysis which revealed that "misspelled" words are actually domain-specific medical terms. This pointed out that the correction of such words would risk retrieval quality, which eventually was not utilized in our research. 

3. **Indexing the Corpus**
   We then index the full document corpus using **PyTerrier**, combining each paper's title and abstract into a single searchable text field.

4. **Relevance Judgments Analysis**
   Lastly, we perform an analysis on the two versions of the relevance judgments, comparing the complete dataset with the Round 5 dataset.

In [1]:
import pyterrier as pt
import os
from pathlib import Path
import spacy
import pandas as pd
import warnings
import nltk
from nltk.stem import PorterStemmer
from spellchecker import SpellChecker
from nltk.corpus import stopwords
from nltk import pos_tag
import re

warnings.filterwarnings('ignore', category=DeprecationWarning)

In [2]:
dataset_name = "irds:cord19/trec-covid"
dataset = pt.get_dataset(dataset_name)

print(f"Loaded dataset: {dataset_name}")
irds_dataset = dataset.irds_ref()

Loaded dataset: irds:cord19/trec-covid


## Query Exploration
### Look at Verbosity Levels

In [3]:
topics_data = []
pd.set_option('display.max_colwidth', None)
for query in irds_dataset.queries_iter():
    topics_data.append({
        'title': query.title,
        'description': query.description,
        'narrative': query.narrative
    })

df_topics = pd.DataFrame(topics_data)
print(f"Total number of queries: {len(df_topics)}")
df_topics.head(1)

Total number of queries: 50


,title,description,narrative
0,coronavirus origin,what is the origin of COVID-19,"seeking range of information about the SARS-CoV-2 virus's origin, including its evolution, animal source, and first transmission into humans"


### Medical Term Analysis using SciSpacy

In [6]:
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
nlp = spacy.load("en_core_sci_sm")

def count_medical_terms(text):
    doc = nlp(text)
    return len(doc.ents) 

df_topics['title_med_count'] = df_topics['title'].apply(count_medical_terms)
df_topics['desc_med_count'] = df_topics['description'].apply(count_medical_terms)
df_topics['narr_med_count'] = df_topics['narrative'].apply(count_medical_terms)

print("Avg Medical Terms in Title:", df_topics['title_med_count'].mean())
print("Avg Medical Terms in Desc:", df_topics['desc_med_count'].mean())
print("Avg Medical Terms in Narr:", df_topics['narr_med_count'].mean())

C:\Users\zosia\anaconda3\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Avg Medical Terms in Title: 2.22
Avg Medical Terms in Desc: 3.52
Avg Medical Terms in Narr: 8.24


In [8]:
# Checking the actual medical terms found in a few sample queries to verify the correctness
def extract_medical_terms(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents]

df_topics['title_med_words'] = df_topics['title'].apply(extract_medical_terms)
df_topics['desc_med_words'] = df_topics['description'].apply(extract_medical_terms)
df_topics['narr_med_words'] = df_topics['narrative'].apply(extract_medical_terms)

for index, row in df_topics.head(3).iterrows():
    print(f"========== TOPIC {index + 1} ==========")
    print(f"Original Title: {row['title']}")
    print(f"Found Entities: {row['title_med_words']}\n")
    
    print(f"Original Desc:  {row['description']}")
    print(f"Found Entities: {row['desc_med_words']}\n")
    
    print(f"Original Narr:  {row['narrative']}")
    print(f"Found Entities: {row['narr_med_words']}")
    print("======================================\n")

========== TOPIC 1 ==========
Original Title: coronavirus origin
Found Entities: ['coronavirus', 'origin']

Original Desc:  what is the origin of COVID-19
Found Entities: ['origin', 'COVID-19']

Original Narr:  seeking range of information about the SARS-CoV-2 virus's origin, including its evolution, animal source, and first transmission into humans
Found Entities: ['information', "SARS-CoV-2 virus's", 'origin', 'evolution', 'animal source', 'transmission', 'humans']

========== TOPIC 2 ==========
Original Title: coronavirus response to weather changes
Found Entities: ['coronavirus', 'response', 'weather', 'changes']

Original Desc:  how does the coronavirus respond to changes in the weather
Found Entities: ['coronavirus', 'changes', 'weather']

Original Narr:  seeking range of information about the SARS-CoV-2 virus viability in different weather/climate conditions as well as information related to transmission of the virus in different climate conditions
Found Entities: ['information'

### Average Length of the Query at each Verbosity Level Before and After Stopword Removal

In [9]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def count_words(text):
    if pd.isna(text): return 0
    return len(str(text).split())

def count_words_without_stopwords(text):
    if pd.isna(text): return 0
    words = str(text).lower().split()
    filtered_words = [w for w in words if w not in stop_words]
    return len(filtered_words)

In [10]:
verbosity_levels = ['title', 'description', 'narrative']

for level in verbosity_levels:
    df_topics[f'{level}_len'] = df_topics[level].apply(count_words)
    df_topics[f'{level}_len_no_stops'] = df_topics[level].apply(count_words_without_stopwords)


print("--- Average Query Length (INCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len = df_topics[f'{level}_len'].mean()
    print(f"{level.capitalize()}: {avg_len:.2f} words")

print("\n--- Average Query Length (EXCLUDING Stopwords) ---")
for level in verbosity_levels:
    avg_len_no_stops = df_topics[f'{level}_len_no_stops'].mean()
    print(f"{level.capitalize()}: {avg_len_no_stops:.2f} words")

--- Average Query Length (INCLUDING Stopwords) ---
Title: 3.24 words
Description: 10.60 words
Narrative: 23.52 words

--- Average Query Length (EXCLUDING Stopwords) ---
Title: 2.74 words
Description: 5.56 words
Narrative: 14.74 words


## Relevance Judgments Exploration

In [11]:
qrels = dataset.get_qrels()

print(f"Total number of relevance judgments: {len(qrels)}")

print("\nRelevance Score Distribution:")
qrels['label'].value_counts()

Total number of relevance judgments: 69318

Relevance Score Distribution:


label
 0    42652
 2    15609
 1    11055
-1        2
Name: count, dtype: int64

## Document Exploration

In [12]:
doc_iter = irds_dataset.docs_iter()

sample_docs = []
for i, doc in enumerate(doc_iter):
    if i >= 5:
        break
    sample_docs.append({
        'docno': doc.doc_id,
        'title': doc.title,
        'abstract': doc.abstract[:300]
    })

df_docs = pd.DataFrame(sample_docs)
df_docs

,docno,title,abstract
0,ug7v899j,"Clinical features of culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, Saudi Arabia","OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, Saudi Arabia. METHODS: Patients with positive M. pneumoniae cultures from respiratory specime"
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in lung disease?,"Inflammatory diseases of the respiratory tract are commonly associated with elevated production of nitric oxide (NO•) and increased indices of NO• -dependent oxidative stress. Although NO• is known to have anti-microbial, anti-inflammatory and anti-oxidant properties, various lines of evidence suppo"
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,"Surfactant protein-D (SP-D) participates in the innate response to inhaled microorganisms and organic antigens, and contributes to immune and inflammatory regulation within the lung. SP-D is synthesized and secreted by alveolar and bronchiolar epithelial cells, but is also expressed by epithelial ce"
3,2b73a28n,Role of endothelin-1 in lung disease,"Endothelin-1 (ET-1) is a 21 amino acid peptide with diverse biological activity that has been implicated in numerous diseases. ET-1 is a potent mitogen regulator of smooth muscle tone, and inflammatory mediator that may play a key role in diseases of the airways, pulmonary circulation, and inflammat"
4,9785vg6d,Gene expression in epithelial cells in response to pneumovirus infection,"Respiratory syncytial virus (RSV) and pneumonia virus of mice (PVM) are viruses of the family Paramyxoviridae, subfamily pneumovirus, which cause clinically important respiratory infections in humans and rodents, respectively. The respiratory epithelial target cells respond to viral infection with s"


# Indexing

In [16]:
index_dir_path = Path("../index/trec_covid_index").resolve()
index_dir = str(index_dir_path)
index_properties_file = os.path.join(index_dir, "data.properties")

In [17]:
def trec_covid_corpus_generator():
    for doc in dataset.get_corpus_iter():
        title = str(doc.get('title', ''))
        abstract = str(doc.get('abstract', ''))
        
        yield {
            'docno': doc['docno'],
            'text': title + " " + abstract
        }

In [18]:
if os.path.exists(index_properties_file):
    print(f"Index found at {index_dir}. Loading it...")
    index_ref = pt.IndexRef.of(index_properties_file)
else:
    print(f"Index not found at {index_dir}. Building it now (this may take a few minutes)...")
    
    indexer = pt.IterDictIndexer(index_dir, blocks=True, meta={'docno': 50})
    index_ref = indexer.index(trec_covid_corpus_generator())
    print("Index built successfully.")

Index found at C:\Users\zosia\Desktop\Studies\Information Retrieval\IR_Project\index\trec_covid_index. Loading it...


In [19]:
#performing preprocessing on the queries
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')

stemmer = PorterStemmer()
spell = SpellChecker()
stop_words = set(stopwords.words('english'))

def preprocess_text(text, level):
    if pd.isna(text):
        return ""
    
    # lowercasing
    text = text.lower()
    # remove punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # level 0: just lowercase + remove punctuation
    if level == 'title':
        return text

    words = text.split()

    # level 1: POS filtering + stemming + stopword removal
    if level == 'description':
        pos_tags = pos_tag(words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    # level 2: spell correction + POS filtering + stemming + stopword removal
    if level == 'narrative':
        corrected_words = [spell.correction(token) or token for token in words]
        pos_tags = pos_tag(corrected_words)
        allowed_pos_tags = {'NN', 'NNS', 'VBN', 'JJ', 'RB', 'NNP', 'NNPS'}
        filtered_pos_words = [token for token, pos in pos_tags if pos in allowed_pos_tags]
        stemmed_words = [stemmer.stem(token) for token in filtered_pos_words]
        filtered_words = [token for token in stemmed_words if token not in stop_words]

    return ' '.join(filtered_words)

print(verbosity_levels)

for level in verbosity_levels:
    print(f"\nPreprocessing with verbosity level: {level}")
    df_topics[f'preprocessed_title_{level}'] = df_topics['title'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_description_{level}'] = df_topics['description'].apply(lambda x: preprocess_text(x, level))
    df_topics[f'preprocessed_narrative_{level}'] = df_topics['narrative'].apply(lambda x: preprocess_text(x, level))

print(df_topics.head(3))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\zosia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\zosia\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\zosia\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


['title', 'description', 'narrative']

Preprocessing with verbosity level: title

Preprocessing with verbosity level: description

Preprocessing with verbosity level: narrative
                                     title  \
0                       coronavirus origin   
1  coronavirus response to weather changes   
2                     coronavirus immunity   

                                                                      description  \
0                                                  what is the origin of COVID-19   
1                      how does the coronavirus respond to changes in the weather   
2  will SARS-CoV2 infected people develop immunity? Is cross protection possible?   

                                                                                                                                                                                               narrative  \
0                                                           seeking range of information abou

In [20]:
#checking the effect of preprocessing on a sample query

sample = df_topics['title'][0]
print("Original:  ", sample)
print("Level 0:   ", preprocess_text(sample, 'title'))  # just lowercase + remove punctuation
print("Level 1:   ", preprocess_text(sample, 'description'))
print("Level 2:   ", preprocess_text(sample, 'narrative'))
print(preprocess_text("coronavyrus originn", 'description'))  # no spell correction
print(preprocess_text("coronavyrus originn", 'narrative'))  # spell correction applied first

Original:   coronavirus origin
Level 0:    coronavirus origin
Level 1:    coronaviru origin
Level 2:    coronaviru origin
coronavyru originn
coronavyru origin


## Spell Checking Analysis

In [21]:
spell = SpellChecker()
def count_spell_errors(text):
    if pd.isna(text): return 0
    text = str(text).lower()
    words = text.split()
    misspelled = spell.unknown(words)
    return len(misspelled)


In [22]:
spell = SpellChecker()

verbosity_levels = ['title', 'description', 'narrative']

print("--- Spell Checking Analysis ---\n")

for level in verbosity_levels:
    all_text = " ".join(df_topics[level].dropna().astype(str).tolist()).lower()
    
    words = re.findall(r'\w+', all_text)
    
    flagged_words = spell.unknown(words)
    
    print(f"[{level.capitalize()} Queries]")
    print(f"Total unique flagged words: {len(flagged_words)}")
    
    if len(flagged_words) > 0:
        sample = list(flagged_words)[:20]
        print(f"Sample of flagged words: {', '.join(sample)}\n")

--- Spell Checking Analysis ---

[Title Queries]
Total unique flagged words: 10
Sample of flagged words: repurposing, cov, cytokine, covid, biomarkers, remdesivir, datasets, coronavirus, hydroxychloroquine, mrna

[Description Queries]
Total unique flagged words: 15
Sample of flagged words: cov, cytokine, covid, triaging, biomarkers, remdesivir, angiotensin, ncov, repurposed, cov2, datasets, underreporting, coronavirus, hydroxychloroquine, mrna

[Narrative Queries]
Total unique flagged words: 15
Sample of flagged words: cov, cytokine, ace2, etc, covid, biomarkers, remdesivir, datasets, angiotensin, cryo, cov2, coronavirus, hydroxychloroquine, r0, mrna



In [23]:
spell = SpellChecker()

verbosity_levels = ['title', 'description', 'narrative']

print("--- Spell Checking Analysis (With Suggested Corrections) ---\n")

for level in verbosity_levels:
    all_text = " ".join(df_topics[level].dropna().astype(str).tolist()).lower()
    
    words = re.findall(r'\w+', all_text)
    
    flagged_words = spell.unknown(words)
    
    print(f"[{level.capitalize()} Queries]")
    print(f"Total unique flagged words: {len(flagged_words)}")
    
    if len(flagged_words) > 0:
        print("Sample of flagged words -> Suggested correction:")
        sample = list(flagged_words)[:20]
        
        for word in sample:
            suggestion = spell.correction(word)
            print(f"  {word:<15} ->  {suggestion}")
            
    print("-" * 40 + "\n")

--- Spell Checking Analysis (With Suggested Corrections) ---

[Title Queries]
Total unique flagged words: 10
Sample of flagged words -> Suggested correction:
  repurposing     ->  purposing
  cov             ->  cop
  cytokine        ->  cytosine
  covid           ->  covin
  biomarkers      ->  bookmarkers
  remdesivir      ->  None
  datasets        ->  None
  coronavirus     ->  None
  hydroxychloroquine ->  None
  mrna            ->  mana
----------------------------------------

[Description Queries]
Total unique flagged words: 15
Sample of flagged words -> Suggested correction:
  cov             ->  cop
  cytokine        ->  cytosine
  covid           ->  covin
  triaging        ->  trialing
  biomarkers      ->  bookmarkers
  remdesivir      ->  None
  angiotensin     ->  None
  ncov            ->  nov
  repurposed      ->  purposed
  cov2            ->  cove
  datasets        ->  None
  underreporting  ->  None
  coronavirus     ->  None
  hydroxychloroquine ->  None
  mrna    

# Relevance Judgements Analysis - complete vs Round 5

In [24]:
if not pt.started():
    pt.init()

dataset_complete = pt.get_dataset("irds:cord19/trec-covid")
dataset_round5 = pt.get_dataset("irds:cord19/trec-covid/round5")

total_docs = dataset_complete.irds_ref().docs_count()
print(f"Total documents in the corpus: {total_docs}")

qrels_complete = dataset_complete.get_qrels()
qrels_round5 = dataset_round5.get_qrels()

graded_complete_count = qrels_complete['docno'].nunique()
graded_round5_count = qrels_round5['docno'].nunique()

ungraded_complete = total_docs - graded_complete_count
ungraded_round5 = total_docs - graded_round5_count

print("---")
print(f"Complete Dataset:")
print(f"  Graded docs:   {graded_complete_count}")
print(f"  Ungraded docs: {ungraded_complete}")
print("---")
print(f"Round 5 Dataset:")
print(f"  Graded docs:   {graded_round5_count}")
print(f"  Ungraded docs: {ungraded_round5}")

Total documents in the corpus: 192509
---
Complete Dataset:
  Graded docs:   37924
  Ungraded docs: 154585
---
Round 5 Dataset:
  Graded docs:   17825
  Ungraded docs: 174684


In [25]:
# Count qrels in complete
qrels_complete_counts = qrels_complete['label'].value_counts().sort_index()
print("Complete Dataset Relevance Label Distribution:")

print(qrels_complete_counts)

# count qrels in round 5
qrels_round5_counts = qrels_round5['label'].value_counts().sort_index()
print("\nRound 5 Dataset Relevance Label Distribution:")
print(qrels_round5_counts)


# print total count of qrels in complete and round 5
total_qrels_complete = len(qrels_complete)
total_qrels_round5 = len(qrels_round5)
print("\nTotal Qrels:")
print(f"Complete Dataset: {total_qrels_complete}")
print(f"Round 5 Dataset: {total_qrels_round5}")

Complete Dataset Relevance Label Distribution:
label
-1        2
 0    42652
 1    11055
 2    15609
Name: count, dtype: int64

Round 5 Dataset Relevance Label Distribution:
label
-1        2
 0    12239
 1     4233
 2     6677
Name: count, dtype: int64

Total Qrels:
Complete Dataset: 69318
Round 5 Dataset: 23151
